# 🏛️ Day 28 Neo4j & LPG 실전 지식 그래프 핸즈온 워크북

> **목적**: 단순한 쿼리 복사-붙여넣기를 넘어, **Neo4j 드라이버 연결 원리**, **LPG(Labeled Property Graph) 4대 구성요소**, **Movies 지식 그래프 탐색**, 그리고 **실무 온라인 서점(Bookstore) 도메인 모델링 설계 및 검증**까지 일관된 흐름으로 정복하는 핸즈온 워크북입니다.

---

## 🗺️ 학습 로드맵
1. **[1단계] 환경 설정 및 싱글톤 드라이버 연결**: `.env` 기반 안전한 세션 풀링 및 연결 검증
2. **[2단계] Movies 데이터셋 상태 확인**: 노드 171개·관계 253개 지식 그래프 검증
3. **[3단계] LPG 4대 핵심 구조 탐색**: 레이블, 방향성 관계, 다중 관계, 관계 속성(`{roles}`, `{rating}`)
4. **[4단계] 파이썬 그래프 분석 & 2-Hop 추천**: 연대별 분포, 다작 배우, 키아누 리브스 공동 출연자
5. **[5단계] 실전 서점 도메인 모델링 & 검증**: 5대 노드·5대 관계 설계 및 4대 비즈니스 질문 도달성 검사
6. **[6단계] 모델링 안티패턴 진단**: 5개 후보 모델 결함 판별 및 시맨틱 방향 역전 오류 분석

---
## 🚀 [1단계] 환경 설정 및 싱글톤 드라이버 연결

- `GraphDatabase.driver()`는 애플리케이션 전체에서 공유하는 커넥션 풀을 관리합니다.
- `.env`에 정의된 로컬 Desktop(`bolt://localhost:7687`) 정보를 `override=True`로 안전하게 로드합니다.

In [ ]:
import os
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv
from neo4j import GraphDatabase

# 환경변수 로드 (메모리 캐시 덮어쓰기를 위해 override=True 적용)
load_dotenv(".env", override=True)
load_dotenv("../.env", override=True)

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 전역 싱글톤 드라이버 생성
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

def run_cypher(query: str, **params) -> list[dict]:
    """Cypher 실행 -> 결과를 dict 리스트로 반환하는 표준 공용 헬퍼."""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

print("✅ Neo4j 드라이버 연결 성공:", NEO4J_URI)

---
## 📊 [2단계] Movies 데이터셋 적재 상태 점검

- Movies 데이터셋은 총 **171개 노드**와 **253개 관계**로 구성된 표준 LPG 예제입니다.
- 노드가 0개인 경우 `data/movies_setup.cypher`를 읽어 자동 적재합니다.

In [ ]:
total_nodes = run_cypher("MATCH (n) RETURN count(n) AS cnt")[0]["cnt"]
print(f"현재 데이터베이스 총 노드 수: {total_nodes}개")

if total_nodes == 0:
    cypher_path = Path("data/movies_setup.cypher")
    if cypher_path.exists():
        print("🔄 Movies 데이터셋을 자동 적재합니다...")
        with open(cypher_path, "r", encoding="utf-8") as f:
            statements = [s.strip() for s in f.read().split(";") if s.strip()]
        with driver.session() as session:
            for stmt in statements:
                session.run(stmt)
        total_nodes = run_cypher("MATCH (n) RETURN count(n) AS cnt")[0]["cnt"]
        print(f"✅ 적재 완료! 총 노드 수: {total_nodes}개")
    else:
        print("⚠️ data/movies_setup.cypher 파일이 없습니다.")
else:
    print("✅ Movies 데이터셋이 정상 적재되어 있습니다.")

---
## 🧱 [3단계] LPG(Labeled Property Graph) 4대 핵심 요소 탐색

1. **노드 & 레이블**: 개체의 종류를 구분하는 태그 (`Movie`, `Person`)
2. **관계와 방향성**: 사실의 인과관계 (`(Person)-[:ACTED_IN]->(Movie)`)
3. **다중 관계**: 동일 개체 쌍에 여러 역할이 공존 (출연과 감독을 동시 수행)
4. **관계 속성**: 상호작용 문맥에 귀속되는 속성 (`roles: ['Neo']`, `rating: 65`)

In [ ]:
# 1. 레이블 목록 및 노드 개수 확인
labels = [r["label"] for r in run_cypher("CALL db.labels() YIELD label")]
movie_cnt = run_cypher("MATCH (m:Movie) RETURN count(m) AS cnt")[0]["cnt"]
person_cnt = run_cypher("MATCH (p:Person) RETURN count(p) AS cnt")[0]["cnt"]

print(f"레이블 목록: {labels}")
print(f"- Movie: {movie_cnt}개 | Person: {person_cnt}개")

# 2. 관계 종류 확인
rel_types = [r["relationshipType"] for r in run_cypher("CALL db.relationshipTypes() YIELD relationshipType")]
print(f"관계 종류 목록: {rel_types}")

In [ ]:
# 3. 다중 관계 탐색: 감독과 출연을 동시에 한 인물과 영화
multi_roles = run_cypher(
    "MATCH (p:Person)-[:ACTED_IN]->(m:Movie), (p)-[:DIRECTED]->(m) "
    "RETURN p.name AS person, m.title AS movie ORDER BY p.name"
)
print("🎬 감독과 출연을 동시에 수행한 인물:")
for row in multi_roles:
    print(f"  • {row['person']} ➡️ '{row['movie']}'")

In [ ]:
# 4. 관계 속성 탐색: 'The Matrix' 출연진의 배역(roles) 속성
matrix_roles = run_cypher(
    "MATCH (p:Person)-[r:ACTED_IN]->(m:Movie {title: 'The Matrix'}) "
    "RETURN p.name AS actor, r.roles AS roles ORDER BY p.name"
)
print("🎭 'The Matrix' 출연진 및 배역(roles 속성):")
for row in matrix_roles:
    print(f"  • {row['actor']:<20} ➡️ {row['roles']}")

---
## 📈 [4단계] 파이썬 데이터 후처리 & 그래프 통계

- Cypher 쿼리로 가져온 그래프 데이터를 파이썬(`Counter`, `pandas`)으로 후처리하여 비즈니스 인사이트를 도출합니다.
- **2-Hop 순회**: 키아누 리브스가 출연한 영화를 함께 찍은 동료 배우 추천

In [ ]:
# 1. 개봉 연대별 영화 수 분포
movies = run_cypher("MATCH (m:Movie) RETURN m.title AS title, m.released AS released")
decades = Counter((m["released"] // 10) * 10 for m in movies if m["released"])
print("📅 연대별 영화 개봉 분포:")
for decade, cnt in sorted(decades.items()):
    print(f"  • {decade}년대: {cnt:>2}편 {'#' * cnt}")

# 2. 최다 출연 배우 TOP 5
actors = run_cypher("MATCH (p:Person)-[:ACTED_IN]->(m:Movie) RETURN p.name AS actor")
top_actors = Counter(r["actor"] for r in actors).most_common(5)
print("
🏆 최다 출연 배우 TOP 5:")
for name, cnt in top_actors:
    print(f"  • {name:<20}: {cnt}편")

# 3. 키아누 리브스와 최다 공동 출연 배우 (2-Hop 패턴)
co_actors = run_cypher(
    "MATCH (p:Person {name: 'Keanu Reeves'})-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(co:Person) "
    "RETURN co.name AS co_actor, count(m) AS shared_cnt "
    "ORDER BY shared_cnt DESC, co_actor LIMIT 5"
)
print("
🤝 키아누 리브스와 최다 공동 출연 배우 (2-Hop 추천):")
for r in co_actors:
    print(f"  • {r['co_actor']:<20}: {r['shared_cnt']}편 공동 출연")

---
## 🛒 [5단계] 실전 온라인 서점(Bookstore) 도메인 그래프 모델링 & 검증

### 📖 비즈니스 시나리오
- 독자(`Reader`)는 책(`Book`)을 구매(`PURCHASED`)하고, 별점(`REVIEWED {rating}`)을 남깁니다.
- 책은 작가(`Author`)가 집필(`WROTE`)하고, 출판사(`Publisher`)가 출간(`PUBLISHED`)합니다.
- 책은 특정 카테고리(`Category`)에 속합니다(`IN_CATEGORY`).

### 🎯 4대 핵심 질문
1. `같은 카테고리 책`: `Book` ➡️ `Category` ➡️ `Book` (2-Hop 왕복)
2. `이 저자의 다른 책`: `Book` ➡️ `Author` ➡️ `Book` (2-Hop 왕복)
3. `이 독자가 산 책`: `Reader` ➡️ `Book` (1-Hop 직접 연결)
4. `이 책 평균 별점`: `Book` ⬅️ `Reader` (별점 속성 보유 관계)

In [ ]:
QUESTIONS = ['같은 카테고리 책', '이 저자의 다른 책', '이 독자가 산 책', '이 책 평균 별점']

# 1. 서점 도메인 모델 스키마 설계
book_model = {
    'nodes': {'Reader', 'Book', 'Author', 'Publisher', 'Category'},
    'relationships': {
        'PURCHASED': ('Reader', 'Book'),
        'REVIEWED': ('Reader', 'Book'),
        'WROTE': ('Author', 'Book'),
        'PUBLISHED': ('Publisher', 'Book'),
        'IN_CATEGORY': ('Book', 'Category')
    },
    'node_properties': {
        'Reader': {'name', 'user_id'},
        'Book': {'title', 'isbn', 'price'},
        'Author': {'name'},
        'Publisher': {'name'},
        'Category': {'name'}
    },
    'rel_properties': {
        'PURCHASED': {'bought_at'},
        'REVIEWED': {'rating', 'comment'},
        'PUBLISHED': {'published_year'}
    }
}

# 2. 4대 질문에 대한 관계 순회 경로
answers = {
    '같은 카테고리 책': ['IN_CATEGORY', 'IN_CATEGORY'],
    '이 저자의 다른 책': ['WROTE', 'WROTE'],
    '이 독자가 산 책': ['PURCHASED'],
    '이 책 평균 별점': ['REVIEWED']
}

def validate_model(model):
    nodes = model.get('nodes', set())
    rels = model.get('relationships', {})
    node_props = model.get('node_properties', {})
    rel_props = model.get('rel_properties', {})
    assert len(nodes) >= 5, '노드 레이블은 5개 이상이어야 합니다'
    assert len(rels) >= 4, '관계를 4개 이상 설계하세요'
    for name, (s, o) in rels.items():
        assert name.replace('_', '').isupper(), f'대문자 스네이크 이름이어야 함: {name}'
        assert s in nodes and o in nodes, f'미등록 레이블: {s}, {o}'
    used = {label for pair in rels.values() for label in pair}
    assert not (nodes - used), f'고립된 노드 존재: {nodes - used}'
    assert any('rating' in props for props in rel_props.values()), '별점은 관계 속성에 있어야 합니다'
    assert not any('rating' in props for props in node_props.values()), '별점이 노드 속성에 있으면 안 됩니다'
    return True

def _walk_ends(rels, chain):
    results = []
    for start in set(rels[chain[0]]):
        curr, path, ok = start, [start], True
        for name in chain:
            l, r = rels[name]
            if curr == l: curr = r
            elif curr == r: curr = l
            else: ok = False; break
            path.append(curr)
        if ok: results.append((start, curr, tuple(path)))
    return results

def check_reachable(model, answers):
    rels = model.get('relationships', {})
    rel_props = model.get('rel_properties', {})
    for q in QUESTIONS:
        chain = answers[q]
        ends = _walk_ends(rels, chain)
        assert ends, f'{q}: 유효하지 않은 경로'
    return True

print("1. 서점 모델 스키마 검증:", "통과 ✅" if validate_model(book_model) else "실패 ❌")
print("2. 4대 비즈니스 질문 도달성 검증:", "통과 ✅" if check_reachable(book_model, answers) else "실패 ❌")

---
## 🔍 [6단계] 모델링 안티패턴 자동 진단

- 5개 후보 모델(A~E)을 검증기에 통과시켜 결함을 진단합니다.
- **주의**: 구조적 검증기를 통과하더라도 시맨틱 관계의 방향이 올바른지(`Author WROTE Book` vs `Book WROTE Author`) 반드시 인간 엔지니어가 검증해야 합니다.

In [ ]:
candidates = {
    'A': book_model,
    'B': {  # 결함: rating을 Book 노드 속성에 둠
        'nodes': {'Reader', 'Book', 'Author', 'Publisher', 'Category'},
        'relationships': {'PURCHASED': ('Reader', 'Book'), 'REVIEWED': ('Reader', 'Book'),
                          'WROTE': ('Author', 'Book'), 'PUBLISHED': ('Publisher', 'Book'),
                          'IN_CATEGORY': ('Book', 'Category')},
        'node_properties': {'Book': {'title', 'rating'}}, 'rel_properties': {},
    },
    'C': {  # 시맨틱 결함: WROTE 방향 역전 (Book -> Author)
        'nodes': {'Reader', 'Book', 'Author', 'Publisher', 'Category'},
        'relationships': {'PURCHASED': ('Reader', 'Book'), 'REVIEWED': ('Reader', 'Book'),
                          'WROTE': ('Book', 'Author'), 'PUBLISHED': ('Publisher', 'Book'),
                          'IN_CATEGORY': ('Book', 'Category')},
        'node_properties': {'Book': {'title'}}, 'rel_properties': {'REVIEWED': {'rating'}},
    },
    'D': {  # 결함: Publisher 노드 고립 (Orphan Node)
        'nodes': {'Reader', 'Book', 'Author', 'Publisher', 'Category'},
        'relationships': {'PURCHASED': ('Reader', 'Book'), 'REVIEWED': ('Reader', 'Book'),
                          'WROTE': ('Author', 'Book'), 'IN_CATEGORY': ('Book', 'Category')},
        'node_properties': {'Book': {'title'}, 'Publisher': {'name'}}, 'rel_properties': {'REVIEWED': {'rating'}},
    },
    'E': {  # 통과: 한글 레이블
        'nodes': {'회원', '도서', '작가', '펴낸곳', '분야'},
        'relationships': {'BOUGHT': ('회원', '도서'), 'RATED': ('회원', '도서'),
                          'AUTHORED': ('작가', '도서'), 'ISSUED': ('펴낸곳', '도서'),
                          'BELONGS_TO': ('도서', '분야')},
        'node_properties': {'도서': {'title'}}, 'rel_properties': {'RATED': {'rating'}, 'BOUGHT': {'bought_at'}},
    }
}

for name, cand in sorted(candidates.items()):
    try:
        validate_model(cand)
        if name == 'C':
            print(f"후보 {name}: ⚠️ 구조 검사 통과, 그러나 시맨틱 오류 ('WROTE' 방향 역전: Book -> Author)")
        else:
            print(f"후보 {name}: ✅ 완전 무결한 표준 모델")
    except AssertionError as err:
        print(f"후보 {name}: ❌ 결함 발견 -> {err}")